# 📊 ML Advanced - XGBoost e LightGBM

## 🎯 Objetivo

Testar algoritmos avançados de Gradient Boosting (XGBoost e LightGBM) para descobrir **exclusivamente o impacto do algoritmo** antes de investir em Feature Engineering V2.

## 🔬 Metodologia

**Dataset:** `workspace.gold.fii_features_v1` (mesmo usado no baseline)  
**Target:** `target_7d` (FII supera IFIX em 7 dias)  
**Split temporal:** Train (2020-2022), Validation (2023), Test (2024-2025)

## ❓ Perguntas-Chave

1. Algum modelo supera o Random Forest (ROC-AUC = 0.6241)?
2. O ganho obtido justifica maior complexidade?
3. Devemos investir em Feature Engineering V2 ou em validação robusta?

## 📋 Baseline Atual

* **Logistic Regression:** ROC-AUC Test = 0.5941
* **Random Forest:** ROC-AUC Test = 0.6241

---

**Importante:** Este notebook **NÃO** cria novas features. Comparação justa entre algoritmos.

In [0]:
%pip install xgboost lightgbm --quiet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [0]:
%sql
-- Carregar dataset Gold V1 (mesmo do baseline)
SELECT *
FROM workspace.gold.fii_features_v1
ORDER BY date, ticker

In [0]:
df = _sqldf.toPandas()

print("=" * 60)
print("DATASET: workspace.gold.fii_features_v1")
print("=" * 60)
print(f"Registros: {len(df):,}")
print(f"Período: {df['date'].min()} até {df['date'].max()}")
print(f"Tickers únicos: {df['ticker'].nunique()}")
print(f"\nTarget: target_7d")
print(f"Distribuição:\n{df['target_7d'].value_counts()}")
print(f"\nFeatures disponíveis: {len(df.columns)}")
print("=" * 60)

In [0]:
# Remover colunas não permitidas
features_to_drop = ['ticker', 'date', 'target_alpha_7d']

X = df.drop(columns=features_to_drop + ['target_7d'])
y = df['target_7d']
dates = df['date']

print("=" * 60)
print("PREPARAÇÃO DOS DADOS")
print("=" * 60)
print(f"Features utilizadas: {X.shape[1]}")
print(f"Registros totais: {len(X):,}")
print(f"\nTarget distribuição:")
print(y.value_counts())
print(f"\nFeatures: {list(X.columns)}")
print("=" * 60)

In [0]:
# Split temporal idêntico ao baseline
dates = pd.to_datetime(dates)
train_mask = (dates >= '2020-03-01') & (dates <= '2022-12-31')
val_mask = (dates >= '2023-01-01') & (dates <= '2023-12-31')
test_mask = (dates >= '2024-01-01') & (dates <= '2025-02-14')

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("=" * 60)
print("SPLIT TEMPORAL")
print("=" * 60)
print(f"\n📅 TRAIN: 2020-03-01 até 2022-12-31")
print(f"   Registros: {len(X_train):,}")
print(f"   Target 1: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")

print(f"\n📅 VALIDATION: 2023-01-01 até 2023-12-31")
print(f"   Registros: {len(X_val):,}")
print(f"   Target 1: {y_val.sum():,} ({y_val.mean()*100:.1f}%)")

print(f"\n📅 TEST: 2024-01-01 até 2025-02-14")
print(f"   Registros: {len(X_test):,}")
print(f"   Target 1: {y_test.sum():,} ({y_test.mean()*100:.1f}%)")

print("\n✅ Sem sobreposição temporal")
print("=" * 60)

---

## 🚀 XGBoost

Gradient Boosting otimizado com paralelização e regularização.

In [0]:
# Hiperparâmetros razoáveis (sem tuning pesado)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

print("Treinando XGBoost...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print("✅ XGBoost treinado")

In [0]:
# Predições
y_val_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred_xgb = xgb_model.predict(X_val)

y_test_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_test_pred_xgb = xgb_model.predict(X_test)

# Métricas
xgb_metrics = {
    'val_auc': roc_auc_score(y_val, y_val_proba_xgb),
    'val_acc': accuracy_score(y_val, y_val_pred_xgb),
    'val_prec': precision_score(y_val, y_val_pred_xgb, zero_division=0),
    'val_rec': recall_score(y_val, y_val_pred_xgb, zero_division=0),
    'val_f1': f1_score(y_val, y_val_pred_xgb, zero_division=0),
    'test_auc': roc_auc_score(y_test, y_test_proba_xgb),
    'test_acc': accuracy_score(y_test, y_test_pred_xgb),
    'test_prec': precision_score(y_test, y_test_pred_xgb, zero_division=0),
    'test_rec': recall_score(y_test, y_test_pred_xgb, zero_division=0),
    'test_f1': f1_score(y_test, y_test_pred_xgb, zero_division=0)
}

print("=" * 60)
print("XGBOOST - RESULTADOS")
print("=" * 60)
print("\n📊 VALIDATION")
print(f"ROC-AUC:   {xgb_metrics['val_auc']:.4f}")
print(f"Accuracy:  {xgb_metrics['val_acc']:.4f}")
print(f"Precision: {xgb_metrics['val_prec']:.4f}")
print(f"Recall:    {xgb_metrics['val_rec']:.4f}")
print(f"F1-Score:  {xgb_metrics['val_f1']:.4f}")

print("\n📊 TEST")
print(f"ROC-AUC:   {xgb_metrics['test_auc']:.4f}")
print(f"Accuracy:  {xgb_metrics['test_acc']:.4f}")
print(f"Precision: {xgb_metrics['test_prec']:.4f}")
print(f"Recall:    {xgb_metrics['test_rec']:.4f}")
print(f"F1-Score:  {xgb_metrics['test_f1']:.4f}")
print("=" * 60)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curve
fpr_val, tpr_val, _ = roc_curve(y_val, y_val_proba_xgb)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba_xgb)

axes[0].plot(fpr_val, tpr_val, label=f'Validation (AUC={xgb_metrics["val_auc"]:.4f})', linewidth=2)
axes[0].plot(fpr_test, tpr_test, label=f'Test (AUC={xgb_metrics["test_auc"]:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('XGBoost - ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confusion Matrix (Test)
cm = confusion_matrix(y_test, y_test_pred_xgb)
ConfusionMatrixDisplay(cm, display_labels=['Não supera', 'Supera']).plot(ax=axes[1], cmap='Blues')
axes[1].set_title('XGBoost - Confusion Matrix (Test)')

plt.tight_layout()
plt.show()

In [0]:
# Top 20 features
feature_importance_xgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(range(len(feature_importance_xgb)), feature_importance_xgb['importance'])
plt.yticks(range(len(feature_importance_xgb)), feature_importance_xgb['feature'])
plt.xlabel('Importance')
plt.title('XGBoost - Top 20 Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Features (XGBoost):")
print(feature_importance_xgb.head(10))

---

## ⚡ LightGBM

Gradient Boosting com algoritmo leaf-wise (mais rápido e eficiente).

In [0]:
# Hiperparâmetros razoáveis (sem tuning pesado)
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
    force_col_wise=True
)

print("Treinando LightGBM...")
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(20, verbose=False)]
)
print("✅ LightGBM treinado")

In [0]:
# Predições
y_val_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]
y_val_pred_lgb = lgb_model.predict(X_val)

y_test_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]
y_test_pred_lgb = lgb_model.predict(X_test)

# Métricas
lgb_metrics = {
    'val_auc': roc_auc_score(y_val, y_val_proba_lgb),
    'val_acc': accuracy_score(y_val, y_val_pred_lgb),
    'val_prec': precision_score(y_val, y_val_pred_lgb, zero_division=0),
    'val_rec': recall_score(y_val, y_val_pred_lgb, zero_division=0),
    'val_f1': f1_score(y_val, y_val_pred_lgb, zero_division=0),
    'test_auc': roc_auc_score(y_test, y_test_proba_lgb),
    'test_acc': accuracy_score(y_test, y_test_pred_lgb),
    'test_prec': precision_score(y_test, y_test_pred_lgb, zero_division=0),
    'test_rec': recall_score(y_test, y_test_pred_lgb, zero_division=0),
    'test_f1': f1_score(y_test, y_test_pred_lgb, zero_division=0)
}

print("=" * 60)
print("LIGHTGBM - RESULTADOS")
print("=" * 60)
print("\n📊 VALIDATION")
print(f"ROC-AUC:   {lgb_metrics['val_auc']:.4f}")
print(f"Accuracy:  {lgb_metrics['val_acc']:.4f}")
print(f"Precision: {lgb_metrics['val_prec']:.4f}")
print(f"Recall:    {lgb_metrics['val_rec']:.4f}")
print(f"F1-Score:  {lgb_metrics['val_f1']:.4f}")

print("\n📊 TEST")
print(f"ROC-AUC:   {lgb_metrics['test_auc']:.4f}")
print(f"Accuracy:  {lgb_metrics['test_acc']:.4f}")
print(f"Precision: {lgb_metrics['test_prec']:.4f}")
print(f"Recall:    {lgb_metrics['test_rec']:.4f}")
print(f"F1-Score:  {lgb_metrics['test_f1']:.4f}")
print("=" * 60)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curve
fpr_val, tpr_val, _ = roc_curve(y_val, y_val_proba_lgb)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba_lgb)

axes[0].plot(fpr_val, tpr_val, label=f'Validation (AUC={lgb_metrics["val_auc"]:.4f})', linewidth=2)
axes[0].plot(fpr_test, tpr_test, label=f'Test (AUC={lgb_metrics["test_auc"]:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('LightGBM - ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confusion Matrix (Test)
cm = confusion_matrix(y_test, y_test_pred_lgb)
ConfusionMatrixDisplay(cm, display_labels=['Não supera', 'Supera']).plot(ax=axes[1], cmap='Greens')
axes[1].set_title('LightGBM - Confusion Matrix (Test)')

plt.tight_layout()
plt.show()

In [0]:
# Top 20 features
feature_importance_lgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(range(len(feature_importance_lgb)), feature_importance_lgb['importance'])
plt.yticks(range(len(feature_importance_lgb)), feature_importance_lgb['feature'])
plt.xlabel('Importance')
plt.title('LightGBM - Top 20 Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Features (LightGBM):")
print(feature_importance_lgb.head(10))

---

## 🏆 Leaderboard Final

Comparação dos 4 modelos testados.

In [0]:
# Resultados do baseline (hardcoded do notebook 32)
baseline_results = {
    'Logistic Regression': {
        'val_auc': 0.5960, 'test_auc': 0.5941,
        'test_acc': 0.5323, 'test_prec': 0.5259, 'test_rec': 0.4873, 'test_f1': 0.5058
    },
    'Random Forest': {
        'val_auc': 0.6295, 'test_auc': 0.6241,
        'test_acc': 0.5607, 'test_prec': 0.5512, 'test_rec': 0.5746, 'test_f1': 0.5626
    }
}

# Compilar todos os resultados
leaderboard = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'ROC-AUC Val': [
        baseline_results['Logistic Regression']['val_auc'],
        baseline_results['Random Forest']['val_auc'],
        xgb_metrics['val_auc'],
        lgb_metrics['val_auc']
    ],
    'ROC-AUC Test': [
        baseline_results['Logistic Regression']['test_auc'],
        baseline_results['Random Forest']['test_auc'],
        xgb_metrics['test_auc'],
        lgb_metrics['test_auc']
    ],
    'Accuracy': [
        baseline_results['Logistic Regression']['test_acc'],
        baseline_results['Random Forest']['test_acc'],
        xgb_metrics['test_acc'],
        lgb_metrics['test_acc']
    ],
    'Precision': [
        baseline_results['Logistic Regression']['test_prec'],
        baseline_results['Random Forest']['test_prec'],
        xgb_metrics['test_prec'],
        lgb_metrics['test_prec']
    ],
    'Recall': [
        baseline_results['Logistic Regression']['test_rec'],
        baseline_results['Random Forest']['test_rec'],
        xgb_metrics['test_rec'],
        lgb_metrics['test_rec']
    ],
    'F1-Score': [
        baseline_results['Logistic Regression']['test_f1'],
        baseline_results['Random Forest']['test_f1'],
        xgb_metrics['test_f1'],
        lgb_metrics['test_f1']
    ]
}).sort_values('ROC-AUC Test', ascending=False)

print("=" * 80)
print("🏆 LEADERBOARD - TODOS OS MODELOS")
print("=" * 80)
print(leaderboard.to_string(index=False))
print("=" * 80)
print(f"\n🥇 VENCEDOR: {leaderboard.iloc[0]['Model']} (ROC-AUC Test = {leaderboard.iloc[0]['ROC-AUC Test']:.4f})")
print("=" * 80)

---

## 🔍 Análise de Feature Importance

Comparação das features mais importantes entre os modelos tree-based.

In [0]:
# Criar DataFrame comparativo
importance_comparison = pd.DataFrame({
    'Feature': X_train.columns,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgb_model.feature_importances_
})

# Calcular média e ordenar
importance_comparison['Mean'] = importance_comparison[['XGBoost', 'LightGBM']].mean(axis=1)
importance_comparison = importance_comparison.sort_values('Mean', ascending=False).head(20)

print("=" * 80)
print("TOP 20 FEATURES - COMPARAÇÃO ENTRE MODELOS")
print("=" * 80)
print(importance_comparison.to_string(index=False))
print("=" * 80)

# Visualização
fig, ax = plt.subplots(figsize=(14, 10))
x_pos = np.arange(len(importance_comparison))
width = 0.35

ax.barh(x_pos - width/2, importance_comparison['XGBoost'], width, label='XGBoost', alpha=0.8)
ax.barh(x_pos + width/2, importance_comparison['LightGBM'], width, label='LightGBM', alpha=0.8)

ax.set_yticks(x_pos)
ax.set_yticklabels(importance_comparison['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - XGBoost vs LightGBM (Top 20)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [0]:
print("=" * 80)
print("🔍 ANÁLISE INTERPRETATIVA DAS FEATURES")
print("=" * 80)

# Identificar tipos de features
top_features = importance_comparison.head(20)['Feature'].tolist()

ifix_features = [f for f in top_features if 'ifix' in f.lower()]
alpha_features = [f for f in top_features if 'alpha' in f.lower()]
div_features = [f for f in top_features if 'div' in f.lower()]
macro_features = [f for f in top_features if any(x in f.lower() for x in ['selic', 'ipca', 'cambio'])]

print(f"\n📊 O IFIX domina os modelos?")
print(f"   Features IFIX no Top 20: {len(ifix_features)} ({len(ifix_features)/20*100:.0f}%)")
if ifix_features:
    print(f"   Exemplos: {', '.join(ifix_features[:3])}")

print(f"\n📈 Alpha é uma feature realmente relevante?")
print(f"   Features Alpha no Top 20: {len(alpha_features)} ({len(alpha_features)/20*100:.0f}%)")
if alpha_features:
    print(f"   Exemplos: {', '.join(alpha_features[:3])}")

print(f"\n💰 Dividendos ajudam?")
print(f"   Features Dividendos no Top 20: {len(div_features)} ({len(div_features)/20*100:.0f}%)")
if div_features:
    print(f"   Exemplos: {', '.join(div_features[:3])}")

print(f"\n🌍 Variáveis macro ajudam?")
print(f"   Features Macro no Top 20: {len(macro_features)} ({len(macro_features)/20*100:.0f}%)")
if macro_features:
    print(f"   Exemplos: {', '.join(macro_features[:3])}")

print(f"\n⚠️ Features potencialmente irrelevantes (baixa importância):")
low_importance = importance_comparison.tail(5)['Feature'].tolist()
for feat in low_importance:
    mean_imp = importance_comparison[importance_comparison['Feature']==feat]['Mean'].values[0]
    print(f"   - {feat}: {mean_imp:.6f}")

print("=" * 80)

---

## 📋 Avaliação Crítica

Respostas às perguntas-chave do projeto.

In [0]:
best_model = leaderboard.iloc[0]
second_model = leaderboard.iloc[1]
rf_auc = baseline_results['Random Forest']['test_auc']

print("=" * 80)
print("📋 AVALIAÇÃO CRÍTICA - RESPOSTAS")
print("=" * 80)

print("\n1️⃣ Algum modelo superou o Random Forest?")
if best_model['Model'] == 'Random Forest':
    print(f"   ❌ NÃO. Random Forest continua sendo o melhor modelo.")
    print(f"   ROC-AUC Test: {best_model['ROC-AUC Test']:.4f}")
else:
    improvement = (best_model['ROC-AUC Test'] - rf_auc) / rf_auc * 100
    print(f"   ✅ SIM. {best_model['Model']} superou o Random Forest.")
    print(f"   ROC-AUC Test: {best_model['ROC-AUC Test']:.4f} (ganho de {improvement:.2f}%)")

print("\n2️⃣ Algum modelo atingiu ROC-AUC > 0.65?")
if best_model['ROC-AUC Test'] > 0.65:
    print(f"   ✅ SIM. {best_model['Model']} atingiu {best_model['ROC-AUC Test']:.4f}")
else:
    print(f"   ❌ NÃO. Melhor resultado: {best_model['ROC-AUC Test']:.4f}")
    print(f"   Ainda existe espaço para melhoria.")

print("\n3️⃣ O ganho obtido justifica maior complexidade?")
if best_model['Model'] in ['XGBoost', 'LightGBM']:
    improvement = best_model['ROC-AUC Test'] - rf_auc
    if improvement > 0.01:  # >1 ponto percentual
        print(f"   ✅ SIM. Ganho de {improvement:.4f} justifica a complexidade.")
    else:
        print(f"   ⚠️ DUVIDOSO. Ganho de apenas {improvement:.4f} pode não justificar.")
else:
    print(f"   ❌ NÃO. Random Forest já é suficiente.")

print("\n4️⃣ Existe evidência de overfitting?")
val_test_diff = abs(best_model['ROC-AUC Val'] - best_model['ROC-AUC Test'])
if val_test_diff > 0.05:  # >5 pontos percentuais
    print(f"   ⚠️ SIM. Diferença Val-Test = {val_test_diff:.4f}")
    print(f"   Modelo pode estar instável.")
else:
    print(f"   ✅ NÃO. Diferença Val-Test = {val_test_diff:.4f} (aceitável)")
    print(f"   Modelo generaliza bem.")

print("\n5️⃣ Qual modelo deve seguir para a próxima fase?")
print(f"   🏆 {best_model['Model']}")
print(f"   ROC-AUC Test: {best_model['ROC-AUC Test']:.4f}")

print("=" * 80)

---

## 🎯 Decisão de Arquitetura

Recomendação do próximo passo do projeto.

In [0]:
best_auc = best_model['ROC-AUC Test']

print("=" * 80)
print("🎯 RECOMENDAÇÃO - PRÓXIMO NOTEBOOK")
print("=" * 80)

if best_auc >= 0.65:
    print("\n✅ RECOMENDAÇÃO: 34_walk_forward_validation")
    print("\n📊 JUSTIFICATIVA:")
    print(f"   - ROC-AUC Test atingiu {best_auc:.4f} (>= 0.65)")
    print(f"   - Modelo {best_model['Model']} está suficientemente forte")
    print(f"   - Priorizar validação robusta antes de criar novas features")
    print(f"   - Walk-forward validation vai testar estabilidade temporal")
    print(f"   - Se validação for positiva → backtest")
    print(f"   - Se validação falhar → considerar Feature Engineering V2")
    
    print("\n🔄 PRÓXIMOS PASSOS:")
    print("   1. 34_walk_forward_validation (validação temporal robusta)")
    print("   2. 35_backtest (simulação vs IFIX)")
    print("   3. Se necessário: Feature Engineering V2")
    
else:
    print("\n🔧 RECOMENDAÇÃO: 34_feature_engineering_v2")
    print("\n📊 JUSTIFICATIVA:")
    print(f"   - ROC-AUC Test = {best_auc:.4f} (< 0.65)")
    print(f"   - Algoritmos avançados não trouxeram ganho significativo")
    print(f"   - Problema está na qualidade das features, não no algoritmo")
    print(f"   - Criar features técnicas (RSI, MAs, beta) pode ser decisivo")
    print(f"   - Testar modelo com features V2 antes de validação temporal")
    
    print("\n🔄 PRÓXIMOS PASSOS:")
    print("   1. 34_feature_engineering_v2 (RSI, MAs, beta, correlações)")
    print("   2. 35_gold_v2 (materializar dataset V2)")
    print("   3. 36_ml_v2 (re-treinar com features V2)")
    print("   4. 37_walk_forward_validation")
    print("   5. 38_backtest")

print("\n" + "=" * 80)
print(f"🏆 MODELO VENCEDOR: {best_model['Model']}")
print(f"📈 ROC-AUC TEST: {best_model['ROC-AUC Test']:.4f}")
print("=" * 80)